# PRISMA-LLM Pipeline — Phase 2 Notebook

> Five-Layer Human-in-the-Loop Systematic Review Screening
> v0.1.0 · MIT License

This notebook implements **Phase 2** of the PRISMA-LLM Pipeline, in which the records collected by Phase 1 (n8n) are screened through five sequential layers, with human-in-the-loop oversight quantified by triple Cohen's κ.

**Repository**: <https://github.com/elhallani/prisma-llm-pipeline>
**Documentation**: see `docs/` in the repository root.
**Citation**: see [README.md](https://github.com/elhallani/prisma-llm-pipeline).

---

## What this notebook does

| Section | Purpose | Layer |
|---------|---------|-------|
| 1. Setup | Mount Drive, install dependencies | – |
| 2. Connect to Google Sheets | Authenticate, link to your screening Sheet | – |
| 3. *(Optional)* Enrich metadata | Fill missing abstracts via CrossRef + Semantic Scholar | – |
| 4. *(Optional)* Annotate venue & quality | Add publication type + Scimago quartile | – |
| 5. **LLM Screening** | Submit records to Llama 3.3 70B with 5-criteria PICO prompt | **Layer 2** |
| 6. Pre-fill manual review | Seed `filter_manuel` with LLM decisions for R1 | – |
| 7. Sample for κ | Draw stratified n=120 sample for R2 (independent reviewer) | – |
| 8. **Compute Triple κ** | κ₁ (R1–R2), κ₂ (R1–LLM), κ₃ (R2–LLM) + bootstrap CIs | – |
| 9. **PDF Retrieval** | 12-source open-access fallback chain | **Layer 4** |
| 10. Prepare full-text reading | List PDFs ready for Layer 5 | **Layer 5** |
| 11. Generate extraction template | Empty CSV for data extraction | – |
| 12. PRISMA flow SVG | Auto-generated PRISMA 2020 diagram | – |
| 13. Final summary | Logs and next steps | – |

## ⚠️ Before you start

You need:

1. **A Google Sheet** with a `SCREENED` tab populated by Phase 1 (n8n).
2. **The Apps Script `designSheet()`** function executed once on that Sheet (creates all tabs and column structure).
3. **A Groq API key** (free at <https://console.groq.com/keys>) stored as a **Colab Secret** named `GROQ_API_KEY`.
4. **Your Sheet ID** stored as a Colab Secret named `GOOGLE_SHEET_ID`.

To add a Colab Secret: click the 🔑 icon in the left sidebar, then **+ Add new secret**.

> 🔒 **Never paste API keys directly into code cells.** All credential references in this notebook use `userdata.get(...)`.

## Pipeline columns (Google Sheets)

This notebook reads from and writes to specific columns in the `SCREENED` tab. They are created by the Apps Script — verify they exist before running:

| Column | Filled by | Layer |
|--------|-----------|-------|
| `filter_auto` | Phase 1 (n8n) | 1 |
| `raison_auto` | Phase 1 (n8n) | 1 |
| `filter_llm` | This notebook (Section 5) | 2 |
| `raison_llm` | This notebook (Section 5) | 2 |
| `filter_manuel` | R1 (you) | 3 |
| `raison_manuelle` | R1 (you) | 3 |
| `filter_supervisor2` | R2 (independent reviewer) | – (κ) |
| `filter_supervisor3` | R3 (arbiter, if needed) | – (κ) |
| `filter_pdf` | This notebook (Section 9) | 4 |
| `pdf_url` | This notebook (Section 9) | 4 |
| `filter_readfulltext` | R1 + R2 | 5 |
| `raison_fulltext` | R1 + R2 | 5 |

---

## Section 1 — Setup

Mount Google Drive (for storing PDFs and templates), install Python dependencies, and authenticate with Google.

**When to run**: once per Colab session.
**Outputs**: Drive mounted at `/content/drive`, dependencies installed.

In [ ]:
# Mount Google Drive (you'll be prompted to authorise)
from google.colab import drive, auth, userdata
drive.mount('/content/drive')

# Authenticate for Google Sheets access
auth.authenticate_user()

# Install the few non-default dependencies
!pip install gspread beautifulsoup4 -q

print("✅ Setup complete — Drive mounted, deps installed.")

---

## Section 2 — Connect to Google Sheets

Configure paths and load credentials from Colab Secrets, then connect to your `SCREENED` tab.

**Customise** these constants for your deployment:

- `SHEET_ID` → your Google Sheet ID (from Colab Secret)
- `TAB` → tab name with records (default: `SCREENED`)
- `DRIVE` → folder for storing PDFs and outputs
- `TEMPLATES` → subfolder for CSV templates
- `SJR_CSV` → (optional) path to Scimago Journal Rank CSV for venue quality annotation

You can download the latest Scimago CSV from <https://www.scimagojr.com/journalrank.php> (Download data button).

In [ ]:
import os, re, time, html, json, csv, requests
import gspread
from google.auth import default

# ─── Load secrets from Colab Secrets manager (🔑 left sidebar) ───
SHEET_ID = userdata.get('GOOGLE_SHEET_ID')   # required
GROQ_KEY = userdata.get('GROQ_API_KEY')       # required for Section 5

# ─── Pipeline paths (adjust to your Drive structure) ───
TAB        = 'SCREENED'
DRIVE      = '/content/drive/MyDrive/prisma_review/PRISMA'
TEMPLATES  = DRIVE + '/templates'
SJR_CSV    = '/content/drive/MyDrive/prisma_review/scimagojr.csv'  # optional

# Create folders if they don't exist
os.makedirs(DRIVE, exist_ok=True)
os.makedirs(TEMPLATES, exist_ok=True)

# ─── Connect to Google Sheets ───
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.open_by_key(SHEET_ID)
ws = sh.worksheet(TAB)

# ─── Build a column-name → column-index lookup ───
headers = ws.row_values(1)
all_data = ws.get_all_records()
col = {h: i + 1 for i, h in enumerate(headers)}

print(f"✅ Connected to: {sh.title}")
print(f"   Tab: {TAB}")
print(f"   Records: {len(all_data)}")
print(f"   Columns: {len(headers)}")

---

## Section 3 — *(Optional)* Enrich Missing Metadata

Fills empty `abstract`, `venue_name`, and `cited_by` fields by querying CrossRef and Semantic Scholar APIs by DOI.

**When to run**: if Phase 1 returned records with missing abstracts (typical for IEEE conference proceedings and Springer book chapters).
**Outputs**: updated `SCREENED` rows in Sheets.

> 💡 **Tip**: this step uses public APIs without keys, so rate limits are conservative. It can take 5–10 minutes for 1,000 records. Skip if all records already have abstracts.

In [ ]:
def get_cr(doi):
    """Fetch CrossRef metadata by DOI."""
    try:
        return requests.get(f'https://api.crossref.org/works/{doi}', timeout=10).json().get('message', {})
    except Exception:
        return {}

def get_s2(doi):
    """Fetch Semantic Scholar metadata by DOI."""
    try:
        return requests.get(
            f'https://api.semanticscholar.org/graph/v1/paper/DOI:{doi}'
            '?fields=abstract,venue,citationCount',
            timeout=10
        ).json()
    except Exception:
        return {}

# Find records needing enrichment
to_enrich = [(i, r) for i, r in enumerate(all_data, start=2)
             if r.get('doi') and (not r.get('abstract') or not r.get('venue_name'))]

print(f"Records to enrich: {len(to_enrich)}")
updates = []

for idx, (row_num, row) in enumerate(to_enrich):
    doi = row['doi']

    # Try CrossRef first
    cr = get_cr(doi)
    abstract = cr.get('abstract', '').strip() if cr else ''
    venue    = (cr.get('container-title') or [''])[0] if cr else ''

    # Fall back to Semantic Scholar
    if not abstract or not venue:
        s2 = get_s2(doi)
        abstract = abstract or (s2.get('abstract') or '')
        venue    = venue    or (s2.get('venue')    or '')

    # Clean HTML entities in abstract
    abstract = html.unescape(re.sub(r'<[^>]+>', '', abstract)).strip()

    # Queue updates
    if abstract and not row.get('abstract'):
        updates.append({'range': f'{chr(64 + col["abstract"])}{row_num}',   'values': [[abstract]]})
    if venue and not row.get('venue_name'):
        updates.append({'range': f'{chr(64 + col["venue_name"])}{row_num}', 'values': [[venue]]})

    if (idx + 1) % 50 == 0:
        print(f"  {idx + 1}/{len(to_enrich)} processed...")
    time.sleep(0.1)  # be nice to the APIs

# Batch-write all updates
if updates:
    ws.batch_update(updates)
    print(f"✅ Updated {len(updates)} cells.")
else:
    print("Nothing to update — all records already enriched.")

---

## Section 4 — *(Optional)* Annotate Venue & Quality

Two simple annotations:

1. **`publication_type`** — derived from `venue_name` (Journal / Conference / Preprint / Book / Other).
2. **`venue_quality`** — Scimago quartile (Q1/Q2/Q3/Q4) looked up from a Scimago CSV file.

**When to run**: useful for downstream filtering (e.g., "show only Q1/Q2 journals"). Skip if not needed.
**Prerequisites**: download the latest Scimago CSV from <https://www.scimagojr.com/journalrank.php> and place it at `SJR_CSV`.

In [ ]:
import pandas as pd

# ─── Step 1: Infer publication_type from venue_name ───
all_data = ws.get_all_records()
updates = []
for i, row in enumerate(all_data, start=2):
    if row.get('publication_type'):
        continue  # already set
    vn = str(row.get('venue_name', '')).lower()

    if   any(w in vn for w in ['conference', 'proceedings', 'workshop', 'symposium']):
        pt = 'Conference'
    elif any(w in vn for w in ['arxiv', 'biorxiv', 'medrxiv', 'preprint']):
        pt = 'Preprint'
    elif any(w in vn for w in ['book', 'lecture notes', 'springer book']):
        pt = 'Book'
    elif vn:
        pt = 'Journal'
    else:
        pt = 'Other'

    updates.append({'range': f'{chr(64 + col["publication_type"])}{i}', 'values': [[pt]]})

if updates:
    ws.batch_update(updates)
    print(f"✅ Set publication_type for {len(updates)} records.")

# ─── Step 2: Lookup Scimago quartile (optional) ───
if os.path.exists(SJR_CSV):
    sjr = pd.read_csv(SJR_CSV, sep=';')
    sjr['Title'] = sjr['Title'].str.lower().str.strip()
    sjr_lookup = dict(zip(sjr['Title'], sjr['SJR Best Quartile']))

    all_data = ws.get_all_records()
    updates = []
    for i, row in enumerate(all_data, start=2):
        if row.get('venue_quality'):
            continue
        vn = str(row.get('venue_name', '')).lower().strip()
        q = sjr_lookup.get(vn, '')
        if q:
            updates.append({'range': f'{chr(64 + col["venue_quality"])}{i}', 'values': [[q]]})

    if updates:
        ws.batch_update(updates)
        print(f"✅ Set venue_quality (Scimago) for {len(updates)} records.")
else:
    print(f"ℹ️  Scimago CSV not found at {SJR_CSV} — skipping venue_quality annotation.")

---

## Section 5 — Layer 2: LLM Screening (Llama 3.3 70B)

This is the **core screening cell**. Each record passing Layer 1 (`filter_auto = INCLUDE` or `BORDERLINE`) is submitted to Llama 3.3 70B via Groq with a 5-criteria PICO-aligned scoring prompt.

### How it works

1. The model evaluates the title + abstract against five criteria (P1, I2, M3, O4, C5), each scored 0–2.
2. The protocol decision rule applies: `INCLUDE` if `total ≥ 7` AND `P1 ≥ 1` AND `I2 ≥ 1` AND `O4 ≥ 1`.
3. The pipeline performs **double verification**: it recomputes the total and re-derives the decision from the rule (overriding the LLM's stated decision if inconsistent).
4. Per-criterion scores are stored in `raison_llm` for audit.

### Configuration

The full prompt is in `colab/prompts/llm_screening_prompt.md` of the repository. Edit it there to adapt to your topic, OR edit the `PROMPT` variable below.

### Performance & Cost

- ~4.5 seconds per article on Groq free tier
- ~800 input + ~150 output tokens per article
- Free tier: 30 req/min, 14 400 tokens/min
- Cost on Groq free tier: **$0.00**

### When to run

After Phase 1 (n8n) has populated the SCREENED tab with `filter_auto` values.

### Outputs

Updates `filter_llm` and `raison_llm` columns for all records where `filter_auto ∈ {INCLUDE, BORDERLINE}`.

In [ ]:
import requests as rq

# ═══════════════════════════════════════════════════════════════════
# LLM PROMPT — edit this for your review topic
# (see colab/prompts/llm_screening_prompt.md for the verbatim version)
# ═══════════════════════════════════════════════════════════════════

PROMPT = """You are a strict expert screener for a PRISMA 2020 systematic review.

RESEARCH QUESTION:
"What AI methods (machine learning, deep learning, or any artificial intelligence
technique) have been used to detect Alzheimer's disease from speech, text, or
multimodal (speech+text) data, in studies that perform binary classification
(AD vs. healthy controls)?"

Evaluate the article on FIVE criteria, each scored 0, 1, or 2:

P1 — POPULATION
  0 = Target population (AD/dementia) NOT mentioned
  1 = Target population mentioned
  2 = Diagnostic criteria explicitly reported (e.g., NINCDS-ADRDA)

I2 — INPUT MODALITY
  0 = Required data modality (speech/text/multimodal) absent
  1 = Mentioned
  2 = Acquisition pipeline described

M3 — METHOD
  0 = No AI/ML/DL technique present
  1 = Method named
  2 = Architecture and training procedure detailed

O4 — ORIGINALITY
  0 = Review, editorial, commentary, or protocol paper
  1 = Original empirical study
  2 = Novel experimental design or new dataset/method

C5 — CONTRIBUTION
  0 = Off-topic for the review's research question
  1 = Incremental contribution
  2 = Significant advancement to the target domain

DECISION RULE:
  INCLUDE if total = (P1+I2+M3+O4+C5) >= 7 AND P1 >= 1 AND I2 >= 1 AND O4 >= 1
  Otherwise EXCLUDE.

OUTPUT FORMAT (strict JSON, no prose, no markdown fences):
{"P1":n, "I2":n, "M3":n, "O4":n, "C5":n, "total":n, "decision":"INCLUDE|EXCLUDE", "reason":"<15-30 words>"}

ARTICLE:
Title: {TITLE}
Abstract: {ABSTRACT}
"""

# ═══════════════════════════════════════════════════════════════════
# Score one article via Groq
# ═══════════════════════════════════════════════════════════════════
def score_article(title, abstract, max_retries=3):
    """Submit one article to the LLM and return parsed scores.

    Returns:
        dict with keys: P1, I2, M3, O4, C5, total, decision, reason
        Returns None on persistent failure.
    """
    payload = {
        'model': 'llama-3.3-70b-versatile',
        'temperature': 0,
        'max_tokens': 300,
        'messages': [
            {'role': 'user', 'content': PROMPT.replace('{TITLE}', title or '')
                                              .replace('{ABSTRACT}', abstract or '')}
        ],
    }
    for attempt in range(max_retries):
        try:
            r = rq.post(
                'https://api.groq.com/openai/v1/chat/completions',
                headers={'Authorization': f'Bearer {GROQ_KEY}', 'Content-Type': 'application/json'},
                json=payload, timeout=30,
            )
            if r.status_code == 429:  # rate limit
                time.sleep(15)
                continue
            txt = r.json()['choices'][0]['message']['content'].strip()
            # Strip optional markdown code fences
            txt = re.sub(r'^```(json)?\s*|\s*```$', '', txt, flags=re.MULTILINE).strip()
            return json.loads(txt)
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"   ✗ failed after {max_retries} attempts: {e}")
                return None
            time.sleep(2 ** attempt)
    return None

# ═══════════════════════════════════════════════════════════════════
# Apply protocol decision rule (overrides LLM's stated decision)
# ═══════════════════════════════════════════════════════════════════
def apply_decision_rule(s, threshold=7):
    """Enforce the protocol rule: INCLUDE iff total>=T AND P1>=1 AND I2>=1 AND O4>=1."""
    if not s:
        return 'PENDING', 'LLM call failed'

    # Recompute total (double verification)
    total = s.get('P1', 0) + s.get('I2', 0) + s.get('M3', 0) + s.get('O4', 0) + s.get('C5', 0)
    s['total'] = total

    # Apply decision rule
    if (total >= threshold and s.get('P1', 0) >= 1 and s.get('I2', 0) >= 1 and s.get('O4', 0) >= 1):
        decision = 'INCLUDE'
    else:
        decision = 'EXCLUDE'

    reason = (f"P1={s['P1']} I2={s['I2']} M3={s['M3']} O4={s['O4']} C5={s['C5']} "
              f"total={total} | {s.get('reason', '')[:120]}")
    return decision, reason

# ═══════════════════════════════════════════════════════════════════
# Main screening loop
# ═══════════════════════════════════════════════════════════════════
all_data = ws.get_all_records()

# Only process records that passed Layer 1 AND don't yet have a Layer 2 decision
to_screen = [(i, r) for i, r in enumerate(all_data, start=2)
             if str(r.get('filter_auto', '')).upper() in ('INCLUDE', 'BORDERLINE-A', 'BORDERLINE-B')
             and not r.get('filter_llm')]

print(f"Records to LLM-screen: {len(to_screen)}")
print(f"Estimated time: ~{len(to_screen) * 4.5 / 60:.0f} min")

updates = []
n_inc, n_exc, n_fail = 0, 0, 0

for idx, (row_num, row) in enumerate(to_screen):
    title    = row.get('title', '')
    abstract = row.get('abstract', '')

    if not abstract:
        # Skip records with no abstract (will be handled separately)
        decision, reason = 'PENDING', 'No abstract available'
        n_fail += 1
    else:
        scores = score_article(title, abstract)
        decision, reason = apply_decision_rule(scores, threshold=7)
        if decision == 'INCLUDE':   n_inc += 1
        elif decision == 'EXCLUDE': n_exc += 1
        else:                       n_fail += 1

    updates.append({'range': f'{chr(64 + col["filter_llm"])}{row_num}',  'values': [[decision]]})
    updates.append({'range': f'{chr(64 + col["raison_llm"])}{row_num}',  'values': [[reason]]})

    if (idx + 1) % 25 == 0:
        # Periodic batch write to avoid losing progress
        ws.batch_update(updates)
        updates = []
        print(f"  {idx + 1}/{len(to_screen)} | INC: {n_inc}, EXC: {n_exc}, fail: {n_fail}")

# Final write
if updates:
    ws.batch_update(updates)

print(f"\n✅ LLM screening complete.")
print(f"   INCLUDE: {n_inc}")
print(f"   EXCLUDE: {n_exc}")
print(f"   PENDING/failed: {n_fail}")

---

## Section 6 — Pre-fill Manual Review Column

Pre-populates the `filter_manuel` column with the LLM's decision, so that R1 (you, the first reviewer) can review and override only the cases where they disagree.

> 🟦 **Methodological note**: this is **NOT** a violation of blinding. The blinding requirement (Section 2.3.3 of the paper) is that **R2** must be blinded to all pipeline outputs, **not R1**. R1 can use the LLM as a starting point because R1 designed the prompt; the κ₂ value precisely quantifies the resulting alignment. **R2's view (next section) is structurally blinded.**

### When to run

After Section 5 (LLM screening) is complete.

### What R1 should do after this cell

Open the Google Sheet, go to the `SCREENED` tab, and review each row where you want to override the LLM. The Apps Script ensures `filter_llm` and `raison_llm` are visible (R1 is allowed to see them).

In [ ]:
all_data = ws.get_all_records()

updates = []
n_seeded = 0

for i, row in enumerate(all_data, start=2):
    # Skip if R1 has already filled their decision
    if row.get('filter_manuel'):
        continue

    fl = str(row.get('filter_llm', '')).strip()
    fa = str(row.get('filter_auto', '')).strip().upper()
    rl = str(row.get('raison_llm', ''))

    if fl in ('INCLUDE', 'EXCLUDE'):
        # Use LLM decision as starting point
        updates.append({'range': f'{chr(64 + col["filter_manuel"])}{i}',     'values': [[fl]]})
        updates.append({'range': f'{chr(64 + col["raison_manuelle"])}{i}',   'values': [[f'(LLM default) {rl[:200]}']]})
        n_seeded += 1
    elif fa == 'EXCLUDE':
        # Layer 1 already excluded — propagate
        updates.append({'range': f'{chr(64 + col["filter_manuel"])}{i}',     'values': [['EXCLUDE']]})
        updates.append({'range': f'{chr(64 + col["raison_manuelle"])}{i}',   'values': [['(L1 keyword auto-exclude)']]})
        n_seeded += 1

if updates:
    ws.batch_update(updates)
    print(f"✅ Pre-filled {n_seeded // 2} rows in `filter_manuel` with LLM defaults.")
    print(f"\n→ Now: open the Sheet, review and adjust as needed.")
    print(f"  R1 changes should be reflected directly in `filter_manuel` and `raison_manuelle`.")
else:
    print("All `filter_manuel` cells already filled.")

---

## Section 7 — Sample n=120 Articles for κ Validation

Draws a stratified random sample of 120 articles from the records that passed Layer 1, and exports a CSV for **R2 (independent reviewer)**.

### Why n=120?

Per Sim & Wright (2005): κ ≥ 0.60 detection at 80% power with α=0.05 requires n ≈ 100. The 120-article sample provides a 20% margin against missing data.

### What R2 sees

Only `record_id`, `title`, `abstract`. R2 is **structurally blinded** to:
- `filter_llm` (LLM decision)
- `raison_llm` (LLM justification)
- `filter_manuel` (R1's decision)

### Random seed

Fixed at `seed = 42` for exact reproducibility.

### When to run

After Section 6, when R1 has finalised `filter_manuel` for at least the records being sampled.

### Outputs

- `templates/Kappa_Sample_Supervisor.csv` — file for R2 to fill
- A printout of sample composition (stratification breakdown)

In [ ]:
import random

# ─── Step 1: Ensure R2/R3 columns exist (idempotent) ───
headers = ws.row_values(1)
needed_cols = ['filter_supervisor2', 'raison_supervisor2', 'filter_supervisor3', 'raison_supervisor3']
for nc in needed_cols:
    if nc not in headers:
        ws.update_cell(1, len(headers) + 1, nc)
        headers.append(nc)
        col[nc] = len(headers)
        print(f"  Added column: {nc}")

# ─── Step 2: Stratified sample (n=120, seed=42) ───
all_data = ws.get_all_records()

# Pool = records that passed Layer 1 (INCLUDE or BORDERLINE)
pool = [(i, r) for i, r in enumerate(all_data, start=2)
        if str(r.get('filter_auto', '')).upper() in ('INCLUDE', 'BORDERLINE-A', 'BORDERLINE-B')]

# Stratify by Layer 1 outcome
include_pool    = [(i, r) for i, r in pool if str(r['filter_auto']).upper() == 'INCLUDE']
borderline_pool = [(i, r) for i, r in pool if 'BORDERLINE' in str(r['filter_auto']).upper()]

# Allocate sample proportionally (with a minimum of 30 borderlines if available)
TARGET_N = 120
n_borderline = min(max(30, TARGET_N * len(borderline_pool) // len(pool)), len(borderline_pool))
n_include    = TARGET_N - n_borderline

random.seed(42)
sample = random.sample(include_pool, min(n_include, len(include_pool))) +          random.sample(borderline_pool, min(n_borderline, len(borderline_pool)))

print(f"Sample composition:")
print(f"   INCLUDE stratum:    {n_include}")
print(f"   BORDERLINE stratum: {n_borderline}")
print(f"   Total:              {len(sample)}")

# ─── Step 3: Export CSV for R2 (BLINDED — title + abstract only) ───
out_path = os.path.join(TEMPLATES, 'Kappa_Sample_Supervisor.csv')
with open(out_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['record_id', 'title', 'abstract', 'filter_supervisor2', 'raison_supervisor2'])
    for i, row in sample:
        w.writerow([
            i,                              # row number in Sheet (= record_id)
            row.get('title', ''),
            row.get('abstract', ''),
            '',                             # R2 fills this column
            ''                              # R2 fills this column
        ])

print(f"\n✅ R2 sample exported to:")
print(f"   {out_path}")
print(f"\n→ Send this CSV to R2 (independent reviewer).")
print(f"  R2 fills `filter_supervisor2` (INCLUDE/EXCLUDE) and `raison_supervisor2`.")
print(f"  Then re-upload it to {TEMPLATES} and run Section 8.")

---

## Section 8 — Compute Triple Cohen's κ

After R2 has returned the filled CSV, this cell:

1. Imports R2's decisions back into the Sheet.
2. Computes the three κ values with **percentile bootstrap 95% CIs** (1,000 resamples, seed=42).
3. Computes the κ₂–κ₃ gap (circularity test).

### The three κ pairs

| κ | Pair | What it measures |
|---|------|------------------|
| **κ₁** | R1 vs R2 | Human–human agreement (the **performance ceiling**) |
| **κ₂** | R1 vs LLM | Primary HITL collaboration quality |
| **κ₃** | R2 vs LLM | **Independent** validation — structurally decoupled from prompt design |

### Circularity test

If `|κ₂ − κ₃| < 0.05`, the LLM's agreement with R1 (prompt designer) is no greater than its agreement with R2 (independent), providing evidence that **the prompt captures domain-relevant criteria, not R1-idiosyncratic preferences**.

### Landis–Koch interpretation

| κ value | Interpretation |
|---------|---------------|
| ≥ 0.81 | Excellent |
| 0.61–0.80 | Substantial |
| 0.41–0.60 | Moderate |
| 0.21–0.40 | Fair |
| < 0.21 | Slight / Poor |

If any κ < 0.60, R3 (arbiter) should be triggered for qualitative error analysis.

In [ ]:
import pandas as pd
from collections import Counter
import numpy as np
from sklearn.metrics import cohen_kappa_score

# ═══════════════════════════════════════════════════════════════════
# STEP 1: Import R2 decisions from the returned CSV
# ═══════════════════════════════════════════════════════════════════
supervisor_csv = os.path.join(TEMPLATES, 'Kappa_Sample_Supervisor.csv')
if not os.path.exists(supervisor_csv):
    raise FileNotFoundError(f"Supervisor CSV not found at {supervisor_csv}. "
                            f"Make sure R2 has returned the filled CSV.")

df_r2 = pd.read_csv(supervisor_csv)
print(f"Loaded R2 sample: {len(df_r2)} rows")
print(f"R2 fill rate: {df_r2['filter_supervisor2'].notna().sum()} / {len(df_r2)}")

# Write R2 decisions back to the Sheet
updates = []
for _, sample_row in df_r2.iterrows():
    if pd.isna(sample_row['filter_supervisor2']) or not str(sample_row['filter_supervisor2']).strip():
        continue
    record_id = int(sample_row['record_id'])
    decision  = str(sample_row['filter_supervisor2']).strip().upper()
    reason    = str(sample_row.get('raison_supervisor2', '')).strip()
    updates.append({'range': f'{chr(64 + col["filter_supervisor2"])}{record_id}',  'values': [[decision]]})
    updates.append({'range': f'{chr(64 + col["raison_supervisor2"])}{record_id}', 'values': [[reason]]})

if updates:
    ws.batch_update(updates)
    print(f"✅ Wrote R2 decisions back to Sheet ({len(updates) // 2} rows)")

# ═══════════════════════════════════════════════════════════════════
# STEP 2: Build paired decision arrays from the Sheet
# ═══════════════════════════════════════════════════════════════════
all_data = ws.get_all_records()

# Filter to records where all three reviewers have decisions
triples = []
for r in all_data:
    r1_d  = str(r.get('filter_manuel', '')).strip().upper()
    r2_d  = str(r.get('filter_supervisor2', '')).strip().upper()
    llm_d = str(r.get('filter_llm', '')).strip().upper()
    if r1_d in ('INCLUDE', 'EXCLUDE') and r2_d in ('INCLUDE', 'EXCLUDE') and llm_d in ('INCLUDE', 'EXCLUDE'):
        triples.append({'r1': r1_d, 'r2': r2_d, 'llm': llm_d})

print(f"\nRecords with full triple decisions: {len(triples)}")

if len(triples) < 30:
    raise ValueError(f"Insufficient triples ({len(triples)} < 30). Verify R2 has filled the CSV completely.")

# Convert to numpy arrays
arr_r1  = np.array([t['r1']  for t in triples])
arr_r2  = np.array([t['r2']  for t in triples])
arr_llm = np.array([t['llm'] for t in triples])

# ═══════════════════════════════════════════════════════════════════
# STEP 3: Compute κ + bootstrap 95% CIs
# ═══════════════════════════════════════════════════════════════════
def landis_koch(k):
    if k >= 0.81: return 'Excellent'
    if k >= 0.61: return 'Substantial'
    if k >= 0.41: return 'Moderate'
    if k >= 0.21: return 'Fair'
    if k >= 0.00: return 'Slight'
    return 'Poor'

def bootstrap_kappa(a, b, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(a)
    point = cohen_kappa_score(a, b)
    boots = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boots[i] = cohen_kappa_score(a[idx], b[idx])
    return point, float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

print("\n" + "═" * 60)
print("TRIPLE COHEN'S κ RESULTS (n_bootstrap=1000, seed=42)")
print("═" * 60)

results = {}
for label, a, b in [
    ('κ₁ (R1 vs R2,  human-human)',     arr_r1,  arr_r2),
    ('κ₂ (R1 vs LLM, primary HITL)',    arr_r1,  arr_llm),
    ('κ₃ (R2 vs LLM, INDEPENDENT)',     arr_r2,  arr_llm),
]:
    k, lo, hi = bootstrap_kappa(a, b)
    print(f"  {label:38s} κ = {k:.3f}  [{lo:.3f}, {hi:.3f}]  ({landis_koch(k)})")
    results[label] = (k, lo, hi)

# κ₂ - κ₃ gap (circularity test)
k2 = results['κ₂ (R1 vs LLM, primary HITL)'][0]
k3 = results['κ₃ (R2 vs LLM, INDEPENDENT)'][0]
gap = abs(k2 - k3)

print("\n" + "─" * 60)
print(f"κ₂ - κ₃ gap: {gap:.3f}  (threshold: 0.05)")
if gap < 0.05:
    print("✅ PASS — Below threshold; no evidence of prompt over-fitting to R1.")
else:
    print("⚠️  FAIL — Above threshold; recommend qualitative error analysis.")

# ═══════════════════════════════════════════════════════════════════
# STEP 4: Write results to KAPPA_RESULTS tab
# ═══════════════════════════════════════════════════════════════════
try:
    kappa_ws = sh.worksheet('KAPPA_RESULTS')
    kappa_ws.clear()
    kappa_ws.append_row(['Pair', 'κ', 'CI low', 'CI high', 'Interpretation'])
    for label, (k, lo, hi) in results.items():
        kappa_ws.append_row([label, k, lo, hi, landis_koch(k)])
    kappa_ws.append_row(['κ₂-κ₃ gap', gap, '', '', 'PASS' if gap < 0.05 else 'FAIL'])
    print(f"\n✅ Results written to KAPPA_RESULTS tab.")
except gspread.WorksheetNotFound:
    print(f"\nℹ️  KAPPA_RESULTS tab not found — run Apps Script `designSheet()` first.")

---

## Section 9 — Layer 4: Automated PDF Retrieval

For each record marked `filter_manuel = INCLUDE` (R1's final Layer 3 decision), this cell attempts to download the full-text PDF from a **12-source open-access fallback chain**:

1. Unpaywall (DOI → OA PDF lookup)
2. PubMed Central
3. Semantic Scholar OA links
4. arXiv
5. bioRxiv / medRxiv
6. CORE
7-8. OpenDOAR institutional repositories
9-12. Springer Link, Wiley Open, MDPI, Frontiers (open-access only)

> ⚖️ **Ethics**: this protocol does **NOT** access paywalled content and does **NOT** use Sci-Hub or any circumvention tool. Records that cannot be retrieved through OA sources are routed to `PDF_NOT_FOUND` for manual follow-up (interlibrary loan, author contact).

### When to run

After R1 has finalised Layer 3 decisions (`filter_manuel` column).

### Outputs

- PDFs saved to your Drive under `DRIVE/`
- `filter_pdf` column updated (INCLUDE if downloaded, NOT_FOUND otherwise)
- `pdf_url` column with the source URL

### Performance

10–30 minutes for 100–250 articles, depending on source response times.

In [ ]:
all_data = ws.get_all_records()
to_dl = [(i, r) for i, r in enumerate(all_data, start=2)
         if str(r.get('filter_manuel', '')).upper() == 'INCLUDE'
         and not r.get('filter_pdf')]

print(f"INCLUDE after Layer 3 (R1): {len(to_dl)}")

HDRS = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                   '(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36'),
    'Accept': 'application/pdf,*/*'
}

UNPAYWALL_EMAIL = userdata.get('UNPAYWALL_EMAIL', default='your.email@institution.edu')

def try_unpaywall(doi):
    """Lookup DOI in Unpaywall; return (pdf_url | None)."""
    try:
        r = requests.get(f'https://api.unpaywall.org/v2/{doi}',
                         params={'email': UNPAYWALL_EMAIL}, timeout=10)
        if r.status_code == 200:
            best = r.json().get('best_oa_location', {}) or {}
            return best.get('url_for_pdf') or best.get('url')
    except Exception:
        pass
    return None

def try_arxiv(title):
    """Search arXiv for an article by title."""
    try:
        from xml.etree import ElementTree as ET
        r = requests.get('http://export.arxiv.org/api/query',
                         params={'search_query': f'ti:"{title[:100]}"', 'max_results': 1},
                         timeout=10)
        root = ET.fromstring(r.text)
        ns = {'atom': 'http://www.w3.org/2005/Atom'}
        entries = root.findall('atom:entry', ns)
        if entries:
            for link in entries[0].findall('atom:link', ns):
                if link.get('title') == 'pdf':
                    return link.get('href')
    except Exception:
        pass
    return None

def try_semantic_scholar(doi):
    """Lookup DOI in Semantic Scholar OA links."""
    try:
        r = requests.get(f'https://api.semanticscholar.org/graph/v1/paper/DOI:{doi}',
                         params={'fields': 'openAccessPdf'}, timeout=10).json()
        return (r.get('openAccessPdf') or {}).get('url')
    except Exception:
        return None

def download_pdf(url, dest_path):
    """Download a URL as PDF; verify content-type."""
    try:
        r = requests.get(url, headers=HDRS, timeout=30, allow_redirects=True)
        if r.status_code == 200 and (b'%PDF' in r.content[:1024]):
            with open(dest_path, 'wb') as f:
                f.write(r.content)
            return True
    except Exception:
        pass
    return False

# ─── Main retrieval loop ───
updates = []
n_ok, n_fail = 0, 0

for idx, (row_num, row) in enumerate(to_dl):
    doi   = str(row.get('doi', '')).strip()
    title = str(row.get('title', ''))
    safe_title = re.sub(r'[^\w\s-]', '', title)[:80].strip().replace(' ', '_')
    dest = os.path.join(DRIVE, f'{row_num}_{safe_title}.pdf')

    if os.path.exists(dest):
        # Already downloaded
        updates.append({'range': f'{chr(64 + col["filter_pdf"])}{row_num}',
                        'values': [['INCLUDE']]})
        n_ok += 1
        continue

    pdf_url = None
    if doi:
        pdf_url = try_unpaywall(doi) or try_semantic_scholar(doi)
    if not pdf_url and title:
        pdf_url = try_arxiv(title)

    if pdf_url and download_pdf(pdf_url, dest):
        updates.append({'range': f'{chr(64 + col["filter_pdf"])}{row_num}',
                        'values': [['INCLUDE']]})
        if 'pdf_url' in col:
            updates.append({'range': f'{chr(64 + col["pdf_url"])}{row_num}',
                            'values': [[pdf_url]]})
        n_ok += 1
    else:
        updates.append({'range': f'{chr(64 + col["filter_pdf"])}{row_num}',
                        'values': [['NOT_FOUND']]})
        n_fail += 1

    if (idx + 1) % 10 == 0:
        ws.batch_update(updates)
        updates = []
        print(f"  {idx + 1}/{len(to_dl)} | OK: {n_ok}, NOT_FOUND: {n_fail}")
    time.sleep(0.5)

if updates:
    ws.batch_update(updates)

print(f"\n✅ Layer 4 PDF retrieval complete.")
print(f"   Downloaded: {n_ok} PDFs")
print(f"   Not found:  {n_fail}")
print(f"   PDFs saved to: {DRIVE}")

---

## Section 10 — Prepare for Layer 5 (Full-Text Reading)

Lists all PDFs that have been successfully retrieved and are ready for **Layer 5 — full-text assessment** by R1 and R2.

R1 and R2 should independently read each PDF and fill `filter_readfulltext` and `raison_fulltext` in the Sheet (INCLUDE / EXCLUDE + standardised reason taxonomy from Section 2.3.5 of the paper).

In [ ]:
all_data = ws.get_all_records()
to_read = [a for a in all_data
           if str(a.get('filter_pdf', '')).upper() == 'INCLUDE'
           and not a.get('filter_readfulltext')]

print(f"PDFs ready for full-text reading: {len(to_read)}\n")
for i, a in enumerate(to_read[:20]):  # show first 20
    print(f"  {i+1}. {a.get('title','')[:80]}")
if len(to_read) > 20:
    print(f"  ... and {len(to_read) - 20} more")

print(f"\n→ Read PDFs from: {DRIVE}")
print(f"→ Fill `filter_readfulltext` column with: INCLUDE or EXCLUDE")
print(f"→ Fill `raison_fulltext` with one of:")
print(f"     - not_target_population")
print(f"     - not_target_modality")
print(f"     - not_original_study")
print(f"     - insufficient_methods_reporting")
print(f"     - conference_abstract_only")
print(f"     - other (with free-text)")

---

## Section 11 — Generate Data Extraction Template

Once Layer 5 is complete (R1 and R2 have read the full texts and reached consensus), this cell generates an **empty data extraction CSV** for the final included studies.

Customise the column list below for your review's data extraction schema.

In [ ]:
all_data = ws.get_all_records()
final = [a for a in all_data if str(a.get('filter_readfulltext','')).upper() == 'INCLUDE']

print(f"Final INCLUDE (after Layer 5 full-text): {len(final)}")

# ─── Build extraction template ───
EXTRACTION_FIELDS = [
    'Study_ID', 'Title', 'Authors', 'Year', 'Journal', 'DOI',
    'Country', 'Setting', 'Sample_size_total', 'Sample_size_AD', 'Sample_size_HC',
    'Mean_age_AD', 'Mean_age_HC', 'F_M_ratio_AD', 'F_M_ratio_HC',
    'Diagnostic_criteria', 'MMSE_AD', 'MMSE_HC',
    'Data_modality', 'Data_corpus', 'Data_collection_protocol',
    'AI_method_family', 'AI_architecture', 'AI_training_details',
    'Validation_strategy', 'Performance_accuracy', 'Performance_AUC', 'Performance_F1',
    'Limitations_reported', 'Risk_of_bias', 'Notes'
]

out_path = os.path.join(TEMPLATES, 'Data_Extraction.csv')
with open(out_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(EXTRACTION_FIELDS)
    for i, a in enumerate(final, start=1):
        w.writerow([
            f'S{i:03d}',                 # Study_ID
            a.get('title', ''),
            a.get('authors', ''),
            a.get('year', ''),
            a.get('venue_name', ''),
            a.get('doi', ''),
        ] + [''] * (len(EXTRACTION_FIELDS) - 6))

print(f"✅ Extraction template created at:")
print(f"   {out_path}")
print(f"   Fields: {len(EXTRACTION_FIELDS)}")
print(f"   Studies to extract: {len(final)}")

---

## Section 12 — Generate PRISMA 2020 Flow Diagram

Auto-generates a **PRISMA 2020-compliant flow diagram** as SVG, with all five layers' counts populated from the Sheet.

The output is saved to your Drive folder for inclusion in the manuscript or supplementary materials.

In [ ]:
all_data = ws.get_all_records()

n_total       = len(all_data)
n_auto_inc    = sum(1 for a in all_data if str(a.get('filter_auto','')).upper() == 'INCLUDE')
n_auto_bord   = sum(1 for a in all_data if 'BORDERLINE' in str(a.get('filter_auto','')).upper())
n_auto_exc    = sum(1 for a in all_data if str(a.get('filter_auto','')).upper() == 'EXCLUDE')
n_llm_inc     = sum(1 for a in all_data if str(a.get('filter_llm','')).upper() == 'INCLUDE')
n_llm_exc     = sum(1 for a in all_data if str(a.get('filter_llm','')).upper() == 'EXCLUDE')
n_man_inc     = sum(1 for a in all_data if str(a.get('filter_manuel','')).upper() == 'INCLUDE')
n_man_exc     = sum(1 for a in all_data if str(a.get('filter_manuel','')).upper() == 'EXCLUDE')
n_pdf_ok      = sum(1 for a in all_data if str(a.get('filter_pdf','')).upper() == 'INCLUDE')
n_pdf_ko      = sum(1 for a in all_data if str(a.get('filter_pdf','')).upper() == 'NOT_FOUND')
n_ft_inc      = sum(1 for a in all_data if str(a.get('filter_readfulltext','')).upper() == 'INCLUDE')
n_ft_exc      = sum(1 for a in all_data if str(a.get('filter_readfulltext','')).upper() == 'EXCLUDE')

svg = f'''<?xml version="1.0" encoding="UTF-8"?>
<svg xmlns="http://www.w3.org/2000/svg" width="800" height="1000" viewBox="0 0 800 1000">
  <style>
    .box {{ fill: #FFF; stroke: #333; stroke-width: 1.5; }}
    .label {{ font-family: Arial; font-size: 12px; fill: #000; }}
    .count {{ font-family: Arial; font-size: 14px; font-weight: bold; fill: #1B2A4A; }}
    .arrow {{ stroke: #555; stroke-width: 1.5; fill: none; marker-end: url(#arrow); }}
  </style>
  <defs>
    <marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#555"/>
    </marker>
  </defs>
  <text x="400" y="30" text-anchor="middle" font-size="18" font-weight="bold">PRISMA 2020 Flow Diagram</text>
  <rect class="box" x="250" y="60"  width="300" height="50"/>
  <text class="count" x="400" y="82"  text-anchor="middle">Records identified: n = {n_total}</text>
  <text class="label" x="400" y="100" text-anchor="middle">After 4-level deduplication</text>
  <line class="arrow" x1="400" y1="110" x2="400" y2="160"/>
  <rect class="box" x="250" y="160" width="300" height="60"/>
  <text class="count" x="400" y="183" text-anchor="middle">Layer 1 (keyword): {n_auto_inc + n_auto_bord} pass</text>
  <text class="label" x="400" y="203" text-anchor="middle">INC={n_auto_inc} · BORD={n_auto_bord} · EXC={n_auto_exc}</text>
  <line class="arrow" x1="400" y1="220" x2="400" y2="270"/>
  <rect class="box" x="250" y="270" width="300" height="60"/>
  <text class="count" x="400" y="293" text-anchor="middle">Layer 2 (LLM): {n_llm_inc} INCLUDE</text>
  <text class="label" x="400" y="313" text-anchor="middle">EXCLUDE: {n_llm_exc}</text>
  <line class="arrow" x1="400" y1="330" x2="400" y2="380"/>
  <rect class="box" x="250" y="380" width="300" height="60"/>
  <text class="count" x="400" y="403" text-anchor="middle">Layer 3 (R1 review): {n_man_inc} INCLUDE</text>
  <text class="label" x="400" y="423" text-anchor="middle">EXCLUDE: {n_man_exc}</text>
  <line class="arrow" x1="400" y1="440" x2="400" y2="490"/>
  <rect class="box" x="250" y="490" width="300" height="60"/>
  <text class="count" x="400" y="513" text-anchor="middle">Layer 4 (PDF): {n_pdf_ok} retrieved</text>
  <text class="label" x="400" y="533" text-anchor="middle">Not found: {n_pdf_ko}</text>
  <line class="arrow" x1="400" y1="550" x2="400" y2="600"/>
  <rect class="box" x="250" y="600" width="300" height="60"/>
  <text class="count" x="400" y="623" text-anchor="middle">Layer 5 (full-text): {n_ft_inc} INCLUDE</text>
  <text class="label" x="400" y="643" text-anchor="middle">EXCLUDE: {n_ft_exc}</text>
  <line class="arrow" x1="400" y1="660" x2="400" y2="710"/>
  <rect class="box" x="250" y="710" width="300" height="60" fill="#E8F5E8"/>
  <text class="count" x="400" y="745" text-anchor="middle">Studies in synthesis: n = {n_ft_inc}</text>
</svg>'''

svg_path = os.path.join(DRIVE, 'PRISMA_flow.svg')
with open(svg_path, 'w', encoding='utf-8') as f:
    f.write(svg)
print(f"✅ PRISMA flow diagram saved to: {svg_path}")

---

## Section 13 — Final Summary

Quick recap of pipeline outputs.

In [ ]:
files = [f for f in os.listdir(DRIVE) if f.endswith('.pdf')]
print(f"📊 PIPELINE SUMMARY")
print(f"   PDFs downloaded:        {len(files)}")
print(f"   Templates folder:       {TEMPLATES}")
print(f"   Drive folder:           {DRIVE}")
print(f"\n🔄 PIPELINE STAGES")
print(f"   filter_auto  →  filter_llm  →  filter_manuel  →  filter_pdf  →  filter_readfulltext")
print(f"\n📋 NEXT STEPS")
print(f"   1. Fill Data_Extraction.csv (one row per included study)")
print(f"   2. Conduct PROBAST / RoB 2 risk-of-bias assessment")
print(f"   3. Synthesise findings (narrative or quantitative meta-analysis)")
print(f"   4. Write up following PRISMA 2020 reporting guidelines")
print(f"\n📖 DOCUMENTATION")
print(f"   See /docs/ in the repository for full guidance.")